In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Image, display
from scipy import stats


In [ ]:
# Download data and build the clean daily BTC/USD dataset

RAW_URL = (
    "https://raw.githubusercontent.com/ff137/"
    "bitstamp-btcusd-minute-data/refs/heads/main/"
    "data/historical/btcusd_bitstamp_1min_2012-2025.csv.gz"
)
ANALYSIS_START = "2015-01-10"
TRAIN_START, TRAIN_END = "2015-01-10", "2019-12-31"
VALIDATION_START, VALIDATION_END = "2020-01-01", "2021-12-31"
TEST_START, TEST_END = "2022-01-01", "2025-01-06"
PERIODS = {
    "train": (TRAIN_START, TRAIN_END),
    "validation": (VALIDATION_START, VALIDATION_END),
    "test": (TEST_START, TEST_END),
    "full": (ANALYSIS_START, TEST_END),
}
HORIZONS = [1, 2, 3, 5, 10, 15, 20, 30, 45, 60, 90, 120]
CANDIDATE_HORIZONS = [5, 10, 15, 20, 30, 45, 60, 90]
MARKET_ORDER_COST = 0.0020
LIMIT_ORDER_COST = 0.0007
ANNUALIZATION = 365
COURSE_ANNUALIZATION = 252

WORK_DIR = Path.cwd() / "notebook_runtime"
WORK_DIR.mkdir(exist_ok=True)
FIGURES_DIR = WORK_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
RAW_PATH = WORK_DIR / "btcusd_bitstamp_1min_2012-2025.csv.gz"
DAILY_PATH = WORK_DIR / "btc_daily.csv"

existing_sources = [
    Path.cwd() / "data" / "raw" / RAW_PATH.name,
    Path.cwd().parent / "data" / "raw" / RAW_PATH.name,
    RAW_PATH,
]
source = next((path for path in existing_sources if path.exists()), None)
if source is None:
    print("Downloading BTC/USD minute data. This is about 95 MB compressed.")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)
    source = RAW_PATH
else:
    print(f"Using raw data source: {source}")

if not DAILY_PATH.exists():
    raw = pd.read_csv(source, compression="gzip")
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], unit="s", utc=True)
    raw = raw.set_index("timestamp").sort_index()
    if raw.index.has_duplicates:
        raise ValueError("Raw data contains duplicate timestamps.")
    if raw[["open", "high", "low", "close", "volume"]].isna().any().any():
        raise ValueError("Raw data contains missing OHLCV values.")

    daily_all = raw.resample("1D").agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume=("volume", "sum"),
        minute_count=("close", "size"),
    )
    daily_clean = daily_all[
        (daily_all["minute_count"] == 1440) & (daily_all["volume"] > 0)
    ].drop(columns="minute_count")
    daily_clean.index.name = "timestamp"
    daily_clean.to_csv(DAILY_PATH)
    print(f"Saved clean daily data: {DAILY_PATH}")

daily = pd.read_csv(DAILY_PATH, parse_dates=["timestamp"]).set_index("timestamp")
daily = daily.loc[ANALYSIS_START:TEST_END].copy()
daily["daily_return"] = daily["close"].pct_change()
print(f"Daily rows used for analysis: {len(daily):,}")
daily.head()


In [ ]:
# Helper functions used by the full notebook backtest

def period_slice(series_or_frame, period):
    start, end = PERIODS[period]
    return series_or_frame.loc[start:end]

def period_labels(index):
    labels = pd.Series("outside", index=index, dtype="object")
    for period in ("train", "validation", "test"):
        start, end = PERIODS[period]
        labels.loc[start:end] = period
    return labels

def volume_zscore(volume, horizon):
    recent_average = volume.rolling(horizon, min_periods=horizon).mean()
    trailing_mean = volume.rolling(120, min_periods=120).mean()
    trailing_std = volume.rolling(120, min_periods=120).std()
    return (recent_average - trailing_mean) / trailing_std.replace(0, np.nan)

def strategy_returns(df, horizon, position_mode, activity_filter, execution_cost=MARKET_ORDER_COST):
    close = df["close"]
    volume = df["volume"]
    daily_return = close.pct_change()
    momentum = np.log(close / close.shift(horizon))

    if position_mode == "long_only":
        raw_position = (momentum > 0).astype(float)
    elif position_mode == "long_short":
        raw_position = np.sign(momentum).fillna(0.0)
    else:
        raise ValueError(position_mode)

    if activity_filter:
        vol_z = volume_zscore(volume, horizon)
        running_median = vol_z.expanding(min_periods=120).median()
        activity_ok = (vol_z > running_median).astype(float)
    else:
        activity_ok = pd.Series(1.0, index=df.index)

    position = (raw_position * activity_ok).shift(1).fillna(0.0)
    turnover = position.diff().abs().fillna(position.abs())
    cost = turnover * execution_cost
    net_return = position * daily_return - cost
    return pd.DataFrame({
        "position": position,
        "turnover": turnover,
        "gross_return": position * daily_return,
        "cost": cost,
        "net_return": net_return,
        "benchmark_return": daily_return,
    }).dropna()

def performance_stats(returns, benchmark=None):
    joined = pd.DataFrame({"returns": returns})
    if benchmark is not None:
        joined["benchmark"] = benchmark
    joined = joined.replace([np.inf, -np.inf], np.nan).dropna()
    rets = joined["returns"]
    wealth = (1.0 + rets).cumprod()
    total_return = wealth.iloc[-1] - 1.0
    cagr = (1.0 + total_return) ** (ANNUALIZATION / len(rets)) - 1.0
    annualized_mean = rets.mean() * ANNUALIZATION
    annualized_vol = rets.std(ddof=1) * np.sqrt(ANNUALIZATION)
    sharpe = rets.mean() / rets.std(ddof=1) * np.sqrt(ANNUALIZATION)
    sharpe_252 = rets.mean() / rets.std(ddof=1) * np.sqrt(COURSE_ANNUALIZATION)
    drawdown = wealth / wealth.cummax() - 1.0
    result = {
        "n_obs": int(len(rets)),
        "total_return": float(total_return),
        "cagr": float(cagr),
        "annualized_mean_return": float(annualized_mean),
        "annualized_volatility": float(annualized_vol),
        "sharpe_ratio": float(sharpe),
        "sharpe_ratio_252": float(sharpe_252),
        "max_drawdown": float(drawdown.min()),
        "alpha_annualized": np.nan,
        "alpha_t_hac": np.nan,
        "alpha_p_hac": np.nan,
        "beta": np.nan,
        "btc_correlation": np.nan,
        "r_squared": np.nan,
    }
    if benchmark is not None:
        x = sm.add_constant(joined["benchmark"])
        model = sm.OLS(rets, x).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        result.update({
            "alpha_annualized": float(model.params.iloc[0] * ANNUALIZATION),
            "alpha_t_hac": float(model.tvalues.iloc[0]),
            "alpha_p_hac": float(model.pvalues.iloc[0]),
            "beta": float(model.params.iloc[1]),
            "btc_correlation": float(rets.corr(joined["benchmark"])),
            "r_squared": float(model.rsquared),
        })
    return result

def evaluate_period(name, model_name, returns, benchmark):
    return {
        "period": name,
        "model": model_name,
        **performance_stats(period_slice(returns, name), period_slice(benchmark, name)),
    }

def benchmark_period(name, benchmark):
    return {"period": name, "model": "btc_buy_hold", **performance_stats(period_slice(benchmark, name))}

BLUE = "#1F5A94"
GOLD = "#B07A16"
GRAY = "#8A939B"
RED = "#A33A3A"

def save_and_show(fig, filename):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(path)))


In [ ]:
# Strategy #1: BTC Momentum Horizon Scan

rows = []
for horizon in HORIZONS:
    past_return = np.log(daily["close"] / daily["close"].shift(horizon))
    forward_return = np.log(daily["close"].shift(-horizon) / daily["close"])
    sample = pd.DataFrame({"past_return": past_return, "forward_return": forward_return})
    train = period_slice(sample, "train").dropna()
    pearson_r, pearson_p = stats.pearsonr(train["past_return"], train["forward_return"])
    model = sm.OLS(
        train["forward_return"],
        sm.add_constant(train["past_return"]),
    ).fit(cov_type="HAC", cov_kwds={"maxlags": horizon})
    rows.append({
        "period": "train",
        "horizon_days": horizon,
        "n_overlapping": len(train),
        "pearson_corr": pearson_r,
        "pearson_naive_p": pearson_p,
        "hac_slope": model.params.iloc[1],
        "hac_p": model.pvalues.iloc[1],
        "signal": "MOMENTUM" if pearson_r > 0 else "REVERSAL",
    })

horizon_scan = pd.DataFrame(rows)
horizon_scan


In [ ]:
# Horizon scan chart

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = np.where(horizon_scan["pearson_corr"] >= 0, BLUE, RED)
ax.bar(horizon_scan["horizon_days"].astype(str), horizon_scan["pearson_corr"], color=colors)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_title("Training-period momentum/reversal horizon scan")
ax.set_xlabel("Lookback and forward horizon (days)")
ax.set_ylabel("Pearson correlation")
ax.grid(axis="y", alpha=0.25)
ax.text(
    0.99,
    0.02,
    "HAC/FDR significance is reported in the results table; none should be inferred from bar height alone.",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=8,
    color="#555555",
)
save_and_show(fig, "horizon_scan.png")


In [ ]:
# Strategy #2: Activity and Seasonality Checks

def correlation_row(analysis, group, sample, outcome_horizon_days):
    sample = sample.dropna()
    pearson_r, naive_p = stats.pearsonr(sample["signal"], sample["outcome"])
    model = sm.OLS(
        sample["outcome"],
        sm.add_constant(sample["signal"]),
    ).fit(cov_type="HAC", cov_kwds={"maxlags": max(1, outcome_horizon_days)})
    return {
        "analysis": analysis,
        "group": group,
        "signal_horizon_days": 20,
        "outcome_horizon_days": outcome_horizon_days,
        "n_obs": len(sample),
        "pearson_corr": pearson_r,
        "pearson_naive_p": naive_p,
        "hac_slope": model.params.iloc[1],
        "hac_p": model.pvalues.iloc[1],
        "mean_outcome_return": sample["outcome"].mean(),
    }

signal_20 = np.log(daily["close"] / daily["close"].shift(20))
forward_20 = np.log(daily["close"].shift(-20) / daily["close"])
vol_z = volume_zscore(daily["volume"], 20)
running_median = vol_z.expanding(min_periods=120).median()
activity = period_slice(
    pd.concat(
        {"signal": signal_20, "outcome": forward_20, "vol_z": vol_z, "threshold": running_median},
        axis=1,
    ),
    "train",
).dropna()

next_day_return = np.log(daily["close"].shift(-1) / daily["close"])
seasonality = period_slice(
    pd.concat({"signal": signal_20, "outcome": next_day_return}, axis=1),
    "train",
).dropna()
is_weekend = seasonality.index.dayofweek.isin([5, 6])

activity_results = pd.DataFrame([
    correlation_row("activity", "high_activity", activity[activity["vol_z"] > activity["threshold"]], 20),
    correlation_row("activity", "low_activity", activity[activity["vol_z"] <= activity["threshold"]], 20),
    correlation_row("entry_day_seasonality", "weekday", seasonality[~is_weekend], 1),
    correlation_row("entry_day_seasonality", "weekend", seasonality[is_weekend], 1),
])
activity_results


In [ ]:
# Activity and seasonality chart

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
activity_subset = activity_results[activity_results["analysis"] == "activity"]
axes[0].bar(
    ["High activity", "Low activity"],
    activity_subset["pearson_corr"],
    color=[BLUE, GRAY],
)
axes[0].set_title("20-day momentum by activity")
axes[0].set_ylabel("Signal/forward-return correlation")
axes[0].grid(axis="y", alpha=0.25)

seasonality_subset = activity_results[activity_results["analysis"] == "entry_day_seasonality"]
axes[1].bar(
    ["Weekday", "Weekend"],
    seasonality_subset["pearson_corr"],
    color=[BLUE, GOLD],
)
axes[1].set_title("20-day signal vs next-day return")
axes[1].set_ylabel("Correlation")
axes[1].grid(axis="y", alpha=0.25)
save_and_show(fig, "activity_and_seasonality.png")


In [ ]:
# Validation-based strategy selection and full backtest

selection_rows = []
for horizon in CANDIDATE_HORIZONS:
    for position_mode in ("long_only", "long_short"):
        for activity_filter in (False, True):
            bt = strategy_returns(
                daily,
                horizon=horizon,
                position_mode=position_mode,
                activity_filter=activity_filter,
            )
            validation = period_slice(bt, "validation")
            stats_row = performance_stats(validation["net_return"], validation["benchmark_return"])
            selection_rows.append({
                "horizon_days": horizon,
                "position_mode": position_mode,
                "activity_filter": activity_filter,
                "validation_sharpe": stats_row["sharpe_ratio"],
                "validation_cagr": stats_row["cagr"],
                "validation_max_drawdown": stats_row["max_drawdown"],
                "validation_total_return": stats_row["total_return"],
                "validation_turnover": validation["turnover"].mean(),
            })

strategy_selection = pd.DataFrame(selection_rows).sort_values(
    ["validation_sharpe", "validation_turnover"],
    ascending=[False, True],
)
strategy_selection["selected"] = False
strategy_selection.loc[strategy_selection.index[0], "selected"] = True
selected_row = strategy_selection.iloc[0]

selected = {
    "selection_rule": "Highest validation-period Sharpe ratio; lower turnover breaks ties.",
    "validation_period": [VALIDATION_START, VALIDATION_END],
    "test_period": [TEST_START, TEST_END],
    "horizon_days": int(selected_row["horizon_days"]),
    "position_mode": str(selected_row["position_mode"]),
    "activity_filter": bool(selected_row["activity_filter"]),
    "market_order_cost": MARKET_ORDER_COST,
}
selected["selected_model_name"] = (
    f"selected_{selected['horizon_days']}d_{selected['position_mode']}_"
    f"{'activity' if selected['activity_filter'] else 'no_activity_filter'}"
)

selected_bt = strategy_returns(
    daily,
    horizon=selected["horizon_days"],
    position_mode=selected["position_mode"],
    activity_filter=selected["activity_filter"],
)
original_bt = strategy_returns(
    daily,
    horizon=20,
    position_mode="long_short",
    activity_filter=True,
)
benchmark = selected_bt["benchmark_return"]

summary_rows = []
trade_rows = []
for period in ("train", "validation", "test", "full"):
    summary_rows.append(evaluate_period(period, selected["selected_model_name"], selected_bt["net_return"], benchmark))
    summary_rows.append(evaluate_period(period, "original_20d_long_short_activity", original_bt["net_return"], original_bt["benchmark_return"]))
    summary_rows.append(benchmark_period(period, benchmark))
    for model_name, backtest in (
        (selected["selected_model_name"], selected_bt),
        ("original_20d_long_short_activity", original_bt),
    ):
        sample = period_slice(backtest, period)
        trade_days = int((sample["turnover"] > 0).sum())
        years = len(sample) / 365
        trade_rows.append({
            "period": period,
            "model": model_name,
            "average_daily_turnover": sample["turnover"].mean(),
            "annualized_turnover": sample["turnover"].mean() * 365,
            "annualized_turnover_252": sample["turnover"].mean() * 252,
            "trade_days": trade_days,
            "average_trade_days_per_year": trade_days / years,
            "percent_days_in_market": (sample["position"] != 0).mean(),
            "percent_days_short": (sample["position"] < 0).mean(),
            "total_execution_cost": sample["cost"].sum(),
        })

performance_summary = pd.DataFrame(summary_rows)
trade_summary = pd.DataFrame(trade_rows)

cost_rows = []
for label, cost in (
    ("zero_cost", 0.0),
    ("limit_order_7bps", LIMIT_ORDER_COST),
    ("market_order_20bps", MARKET_ORDER_COST),
):
    cost_bt = strategy_returns(
        daily,
        horizon=selected["horizon_days"],
        position_mode=selected["position_mode"],
        activity_filter=selected["activity_filter"],
        execution_cost=cost,
    )
    for period in ("test", "full"):
        cost_rows.append({
            "cost_scenario": label,
            "cost_per_unit_turnover": cost,
            "period": period,
            **performance_stats(period_slice(cost_bt["net_return"], period), period_slice(cost_bt["benchmark_return"], period)),
        })
cost_sensitivity = pd.DataFrame(cost_rows)

curves = pd.DataFrame(index=selected_bt.index)
curves["period"] = period_labels(curves.index)
curves["selected_net_return"] = selected_bt["net_return"]
curves["original_net_return"] = original_bt["net_return"].reindex(curves.index)
curves["buy_hold_return"] = benchmark
for prefix in ("selected", "original", "buy_hold"):
    returns = curves[f"{prefix}_net_return"] if prefix != "buy_hold" else curves["buy_hold_return"]
    wealth = (1.0 + returns.fillna(0.0)).cumprod()
    curves[f"{prefix}_cum"] = wealth
    curves[f"{prefix}_drawdown"] = wealth / wealth.cummax() - 1.0

year_rows = []
for year in sorted(selected_bt.index.year.unique()):
    if year < 2015:
        continue
    year_index = selected_bt.index[selected_bt.index.year == year]
    if len(year_index) < 2:
        continue
    year_start = year_index.min()
    year_end = year_index.max()
    year_period = period_labels(pd.DatetimeIndex([year_start])).iloc[0]
    for model_name, returns, bench in (
        (selected["selected_model_name"], selected_bt["net_return"], benchmark),
        ("btc_buy_hold", benchmark, None),
    ):
        sample_returns = returns.loc[year_start:year_end]
        sample_benchmark = bench.loc[year_start:year_end] if bench is not None else None
        year_rows.append({
            "year": int(year),
            "period_label": year_period,
            "model": model_name,
            **performance_stats(sample_returns, sample_benchmark),
        })
calendar_year_performance = pd.DataFrame(year_rows)

selected


In [ ]:
# Grid Search / Validation Selection

strategy_selection.head(10)


In [ ]:
# Selected strategy backtest

selected


In [ ]:
# Performance summary

performance_summary


In [ ]:
# Held-out test comparison

test = performance_summary[performance_summary["period"] == "test"].copy()
test[["model", "total_return", "cagr", "annualized_volatility", "sharpe_ratio", "sharpe_ratio_252", "max_drawdown", "alpha_annualized", "alpha_t_hac", "alpha_p_hac", "beta", "btc_correlation", "r_squared"]]


In [ ]:
# Exposure and turnover diagnostics

selected_test = test[test["model"].str.startswith("selected")].iloc[0]
selected_trade = trade_summary[
    (trade_summary["period"] == "test")
    & (trade_summary["model"] == selected["selected_model_name"])
].iloc[0]

diagnostics = pd.DataFrame({
    "Diagnostic": [
        "HAC alpha t-stat",
        "HAC alpha p-value",
        "BTC beta",
        "BTC correlation",
        "R-squared",
        "Sharpe, 252-day convention",
        "Annualized turnover",
        "Trade days per year",
    ],
    "Value": [
        f"{selected_test['alpha_t_hac']:.2f}",
        f"{selected_test['alpha_p_hac']:.3f}",
        f"{selected_test['beta']:.2f}",
        f"{selected_test['btc_correlation']:.2f}",
        f"{selected_test['r_squared']:.2f}",
        f"{selected_test['sharpe_ratio_252']:.2f}",
        f"{selected_trade['annualized_turnover']:.2f}",
        f"{selected_trade['average_trade_days_per_year']:.2f}",
    ],
})
diagnostics


In [ ]:
# Calendar-year performance in the held-out test period

test_years = calendar_year_performance[
    calendar_year_performance["period_label"] == "test"
].copy()
test_years[["year", "model", "total_return", "sharpe_ratio", "max_drawdown"]]


In [ ]:
# Formatted summary used in the project PDF and presentation

selected_test = test[test["model"].str.startswith("selected")].iloc[0]
btc_test = test[test["model"] == "btc_buy_hold"].iloc[0]

formatted_summary = pd.DataFrame({
    "Metric": [
        "Total return",
        "CAGR",
        "Annualized volatility",
        "Sharpe ratio",
        "Maximum drawdown",
    ],
    "Selected strategy": [
        f"{selected_test['total_return'] * 100:.1f}%",
        f"{selected_test['cagr'] * 100:.1f}%",
        f"{selected_test['annualized_volatility'] * 100:.1f}%",
        f"{selected_test['sharpe_ratio']:.2f}",
        f"{selected_test['max_drawdown'] * 100:.1f}%",
    ],
    "BTC buy and hold": [
        f"{btc_test['total_return'] * 100:.1f}%",
        f"{btc_test['cagr'] * 100:.1f}%",
        f"{btc_test['annualized_volatility'] * 100:.1f}%",
        f"{btc_test['sharpe_ratio']:.2f}",
        f"{btc_test['max_drawdown'] * 100:.1f}%",
    ],
})
formatted_summary


In [ ]:
# Cost sensitivity

cost_sensitivity


In [ ]:
# Backtest curves

test_curves = curves[curves["period"] == "test"]
strategy = test_curves["selected_cum"] / test_curves["selected_cum"].iloc[0]
benchmark = test_curves["buy_hold_cum"] / test_curves["buy_hold_cum"].iloc[0]
strategy_dd = strategy / strategy.cummax() - 1.0
benchmark_dd = benchmark / benchmark.cummax() - 1.0

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1]},
)
axes[0].plot(test_curves.index, strategy, color=BLUE, label="Selected strategy")
axes[0].plot(test_curves.index, benchmark, color=GRAY, label="BTC buy & hold")
axes[0].set_ylabel("Growth of $1")
axes[0].set_title("Held-out test period: 2022–2025")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.25)
axes[1].plot(test_curves.index, strategy_dd * 100, color=BLUE)
axes[1].plot(test_curves.index, benchmark_dd * 100, color=GRAY)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.25)
save_and_show(fig, "backtest_test_period.png")


## Cross-sectional 17-coin crypto extension

The BTC strategy still has meaningful BTC market exposure, so this notebook also tests a broader dollar-neutral crypto portfolio. The extension downloads Yahoo Finance daily prices for:

BTC-USD, ETH-USD, BNB-USD, XRP-USD, ADA-USD, DOGE-USD, MATIC-USD, LTC-USD, BCH-USD, LINK-USD, XLM-USD, ATOM-USD, ETC-USD, FIL-USD, TRX-USD, EOS-USD, and XMR-USD.

The rule ranks coins by momentum, goes long the strongest three, shorts the weakest three, and keeps net exposure near zero.


In [ ]:
# Cross-sectional 17-coin dollar-neutral momentum extension

MULTICOIN_TICKERS = [
    "BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "ADA-USD",
    "DOGE-USD", "MATIC-USD", "LTC-USD", "BCH-USD", "LINK-USD",
    "XLM-USD", "ATOM-USD", "ETC-USD", "FIL-USD", "TRX-USD",
    "EOS-USD", "XMR-USD",
]
MULTI_PERIODS = {
    "train": ("2020-01-01", "2020-12-31"),
    "validation": ("2021-01-01", "2021-12-31"),
    "test": ("2022-01-01", "2025-01-06"),
    "full": ("2020-01-01", "2025-01-06"),
}
MULTI_PRICES = WORK_DIR / "multicoin_prices.csv"

def unix_time(date):
    return int(pd.Timestamp(date, tz="UTC").timestamp())

def fetch_yahoo_close(ticker):
    url = (
        f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
        f"?period1={unix_time('2020-01-01')}"
        f"&period2={unix_time('2025-01-07')}"
        "&interval=1d&events=history"
    )
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = json.loads(response.read())
    result = payload["chart"]["result"][0]
    dates = pd.to_datetime(result["timestamp"], unit="s", utc=True).tz_convert(None).normalize()
    close = result["indicators"]["quote"][0]["close"]
    return pd.Series(close, index=dates, name=ticker).dropna()

if MULTI_PRICES.exists():
    multicoin_prices = pd.read_csv(MULTI_PRICES, index_col="timestamp", parse_dates=True)
else:
    multicoin_prices = pd.concat(
        [fetch_yahoo_close(ticker) for ticker in MULTICOIN_TICKERS],
        axis=1,
    ).loc["2020-01-01":"2025-01-06"]
    multicoin_prices = multicoin_prices.dropna(axis=1)
    multicoin_prices.index.name = "timestamp"
    multicoin_prices.to_csv(MULTI_PRICES)

def multi_slice(series_or_frame, period):
    start, end = MULTI_PERIODS[period]
    return series_or_frame.loc[start:end]

def multi_labels(index):
    labels = pd.Series("outside", index=index, dtype="object")
    for period in ("train", "validation", "test"):
        start, end = MULTI_PERIODS[period]
        labels.loc[start:end] = period
    return labels

def xs_momentum(prices, lookback, rebalance_days, execution_cost=MARKET_ORDER_COST, top_bottom_count=3):
    returns = prices.pct_change()
    momentum = np.log(prices / prices.shift(lookback))
    raw_weights = pd.DataFrame(np.nan, index=prices.index, columns=prices.columns)
    valid_dates = momentum.dropna(how="any").index
    for date in valid_dates[::rebalance_days]:
        signal = momentum.loc[date].dropna().sort_values()
        shorts = signal.head(top_bottom_count).index
        longs = signal.tail(top_bottom_count).index
        weights = pd.Series(0.0, index=prices.columns)
        weights.loc[longs] = 0.5 / top_bottom_count
        weights.loc[shorts] = -0.5 / top_bottom_count
        raw_weights.loc[date] = weights
    weights = raw_weights.ffill().fillna(0.0).shift(1).fillna(0.0)
    turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    gross_return = (weights * returns).sum(axis=1)
    net_return = gross_return - turnover * execution_cost
    return pd.DataFrame({
        "net_return": net_return,
        "turnover": turnover,
        "gross_exposure": weights.abs().sum(axis=1),
        "net_exposure": weights.sum(axis=1),
        "btc_return": returns["BTC-USD"],
        "equal_weight_return": returns.mean(axis=1),
    }).dropna()

xs_selection_rows = []
for lookback in [7, 14, 30, 60, 90]:
    for rebalance_days in [1, 7, 14, 30]:
        xs_bt = xs_momentum(multicoin_prices, lookback, rebalance_days)
        val = multi_slice(xs_bt, "validation")
        val_stats = performance_stats(val["net_return"], val["btc_return"])
        xs_selection_rows.append({
            "lookback_days": lookback,
            "rebalance_days": rebalance_days,
            "validation_sharpe": val_stats["sharpe_ratio"],
            "validation_turnover": val["turnover"].mean(),
        })

xs_selection = pd.DataFrame(xs_selection_rows).sort_values(
    ["validation_sharpe", "validation_turnover"],
    ascending=[False, True],
)
xs_selected = xs_selection.iloc[0].to_dict()
xs_bt = xs_momentum(
    multicoin_prices,
    int(xs_selected["lookback_days"]),
    int(xs_selected["rebalance_days"]),
)

xs_perf_rows = []
for model_name, rets, btc in (
    ("dollar_neutral_momentum", xs_bt["net_return"], xs_bt["btc_return"]),
    ("equal_weight_crypto", xs_bt["equal_weight_return"], xs_bt["btc_return"]),
    ("btc_buy_hold", xs_bt["btc_return"], None),
):
    xs_perf_rows.append({
        "model": model_name,
        **performance_stats(
            multi_slice(rets, "test"),
            multi_slice(btc, "test") if btc is not None else None,
        ),
    })
xs_test_performance = pd.DataFrame(xs_perf_rows)

xs_trade_days = int((multi_slice(xs_bt, "test")["turnover"] > 0).sum())
xs_years = len(multi_slice(xs_bt, "test")) / 365
xs_summary = {
    "universe_size": len(multicoin_prices.columns),
    "selected_lookback_days": int(xs_selected["lookback_days"]),
    "selected_rebalance_days": int(xs_selected["rebalance_days"]),
    "trade_days_per_year": xs_trade_days / xs_years,
}
xs_summary, xs_test_performance


In [ ]:
# Cross-sectional extension chart

xs_curves = pd.DataFrame(index=xs_bt.index)
xs_curves["period"] = multi_labels(xs_curves.index)
xs_curves["strategy_return"] = xs_bt["net_return"]
xs_curves["equal_weight_return"] = xs_bt["equal_weight_return"]
xs_curves["btc_return"] = xs_bt["btc_return"]
for prefix in ("strategy", "equal_weight", "btc"):
    wealth = (1.0 + xs_curves[f"{prefix}_return"].fillna(0.0)).cumprod()
    xs_curves[f"{prefix}_cum"] = wealth
    xs_curves[f"{prefix}_drawdown"] = wealth / wealth.cummax() - 1.0

xs_test = xs_curves[xs_curves["period"] == "test"]
strategy = xs_test["strategy_cum"] / xs_test["strategy_cum"].iloc[0]
equal_weight = xs_test["equal_weight_cum"] / xs_test["equal_weight_cum"].iloc[0]
btc = xs_test["btc_cum"] / xs_test["btc_cum"].iloc[0]
strategy_dd = strategy / strategy.cummax() - 1.0
equal_weight_dd = equal_weight / equal_weight.cummax() - 1.0
btc_dd = btc / btc.cummax() - 1.0

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1]},
)
axes[0].plot(xs_test.index, strategy, color=BLUE, label="Dollar-neutral momentum")
axes[0].plot(xs_test.index, equal_weight, color=GOLD, label="Equal-weight crypto")
axes[0].plot(xs_test.index, btc, color=GRAY, label="BTC buy & hold")
axes[0].set_ylabel("Growth of $1")
axes[0].set_title("Cross-sectional crypto extension: held-out test period")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.25)
axes[1].plot(xs_test.index, strategy_dd * 100, color=BLUE)
axes[1].plot(xs_test.index, equal_weight_dd * 100, color=GOLD)
axes[1].plot(xs_test.index, btc_dd * 100, color=GRAY)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.25)
save_and_show(fig, "multicoin_backtest_test_period.png")


In [ ]:
# Direct coin-by-coin replication of the BTC rule

individual_rows = []
for coin in multicoin_prices.columns:
    px = multicoin_prices[coin].dropna()
    coin_returns = px.pct_change()
    coin_signal = np.log(px / px.shift(30))
    coin_position = (coin_signal > 0).astype(float).shift(1).fillna(0.0)
    coin_turnover = coin_position.diff().abs().fillna(coin_position.abs())
    coin_net = coin_position * coin_returns - coin_turnover * MARKET_ORDER_COST
    coin_test = multi_slice(coin_net, "test").dropna()
    coin_buy_hold = multi_slice(coin_returns, "test").loc[coin_test.index]
    strategy_stats = performance_stats(coin_test, coin_buy_hold)
    buy_hold_stats = performance_stats(coin_buy_hold, None)
    individual_rows.append({
        "coin": coin,
        "strategy_total_return": strategy_stats["total_return"],
        "strategy_sharpe": strategy_stats["sharpe_ratio"],
        "strategy_max_drawdown": strategy_stats["max_drawdown"],
        "buy_hold_total_return": buy_hold_stats["total_return"],
        "buy_hold_sharpe": buy_hold_stats["sharpe_ratio"],
        "buy_hold_max_drawdown": buy_hold_stats["max_drawdown"],
    })

individual_test = pd.DataFrame(individual_rows).sort_values(
    "strategy_sharpe", ascending=False
)
individual_test


In [ ]:
# Conclusion

print("The final selected rule is a 30-day long-only BTC momentum strategy.")
print("It was profitable in the held-out test period and reduced volatility and drawdown versus BTC buy-and-hold.")
print("It did not beat BTC on total return, and the positive alpha estimate was not statistically significant.")
print("The 17-coin dollar-neutral extension removes BTC exposure, but its after-cost test performance is weak.")
print(f"Applied independently to all 17 coins, the same BTC rule had positive returns for {(individual_test['strategy_total_return'] > 0).sum()}/17 coins and positive Sharpe for {(individual_test['strategy_sharpe'] > 0).sum()}/17 coins.")
